In [1]:
import pandas as pd
import lightgbm as lgb #light gradient boosting this a high performance boosting algorithm in which the decision tree is built sequencially to minimize error
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

feature_dataset = pd.read_csv("../data/processed/feature_dataset.csv")
feature_dataset.head()
print(feature_dataset.shape) #shape helps in creating the output in a format of rows, column
feature_dataset.info()
feature_dataset["SepsisLabel"].value_counts()


(20336, 201)
<class 'pandas.DataFrame'>
RangeIndex: 20336 entries, 0 to 20335
Columns: 201 entries, HR_mean to SepsisLabel
dtypes: float64(194), int64(7)
memory usage: 31.2 MB


SepsisLabel
0    18546
1     1790
Name: count, dtype: int64

In [2]:
X = feature_dataset.drop(columns=["SepsisLabel"])
y = feature_dataset["SepsisLabel"]

print("Feature matrix shape :", X.shape)
print("Target vector shape  :", y.shape)

Feature matrix shape : (20336, 200)
Target vector shape  : (20336,)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y    # stratify is used to preserve the ratio between sepsis and non sepsis in train and test to avoid bias
)
print("Training samples :", X_train.shape)
print("Testing samples  :", X_test.shape) #output denotes the patients and the features (x,y) if the x is the no of patients then the y is  the features each patient have 200 features but same for all patient

model = lgb.LGBMClassifier(
    objective = "binary",
    random_state = 42, # random state is used to train the exact same no of patients during training so tht during changing parameter we can identify whether there model is improved or not if not using random state then model takes it own set so we cant figure out whether accuracy precision increased due to the parameter change or the seed change
)
model.fit(X_train, y_train) # .fit() in this the lgb builds sequence of decision tree sequencially lgbm goes through 16k patients
print("model trained successfully") # so the output the pavg denotes the avg x 100 percentage of sepsis = 8.8 , init score denotes initial prediction before learning.

Training samples : (16268, 200)
Testing samples  : (4068, 200)
[LightGBM] [Info] Number of positive: 1432, number of negative: 14836
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011353 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 31068
[LightGBM] [Info] Number of data points in the train set: 16268, number of used features: 192
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.088026 -> initscore=-2.337985
[LightGBM] [Info] Start training from score -2.337985
model trained successfully


In [5]:
from pathlib import Path
import joblib

model_path = Path("../models")
model_path.mkdir(exist_ok=True)

joblib.dump(model, model_path / "physionet_lightgbm.pkl")

print("LightGBM model saved successfully!")

LightGBM model saved successfully!
